# Clean the Aged Care Timeliness rate ratios and insert into DB

This notebook builds a data pipeline for **Table S2** of the AIHW report
*Timeliness of access to aged care services in Australia* (cat. AGE 118)
and writes one curated table:

- `aged_care_timeliness_prac` — long-format table of event rate ratios for
  the **'PRAC only'** approval group (people approved for permanent
  residential aged care only)

The source xlsx publishes Table S2 as a wide sheet with all three approval
groups side-by-side. This pipeline keeps only the PRAC-only slice — that's
the cohort the calculator scores against.

## Source layout
- Sheet `Table S2`, header rows 2–3, data rows 4–51
- Columns 7, 8, 9 are PRAC only's `Event rate ratio`, `Lower CI`, `Upper CI`
- Category headers (e.g. `Sex`, `Priority level`) sit on their own row with
  blank ratio cells; the rows below them are the values for that category
- Reference categories carry `event_rate_ratio = 1` and blank CIs

## Cleaning rules
- read columns 0, 7, 8, 9 from rows 4–51
- coerce ratio and CI columns to numeric — NaN means "category header"
- forward-fill the category label onto each value row
- drop the category-header rows (they have no ratio)
- mark reference rows where `event_rate_ratio == 1` as `is_reference = True`
- preserve NULL CIs on reference rows — do not fill with 1


In [1]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("input/AIHW-AGE-118-Timeliness-of-access-to-aged-care-services-in-Australia-Supplementary-results-tables_1.xlsx")

INPUT_XLSX

PosixPath('input/AIHW-AGE-118-Timeliness-of-access-to-aged-care-services-in-Australia-Supplementary-results-tables_1.xlsx')

In [2]:
raw = pd.read_excel(INPUT_XLSX, sheet_name="Table S2", header=None)
raw.shape, raw.iloc[2:4, [0, 1, 4, 7]].values.tolist()

((55, 10),
 [[' ',
   "'HCP and PRAC' approval group",
   "'HCP only' approval group",
   "'PRAC only' approval group"],
  ['Covariate', 'Event rate ratio', 'Event rate ratio', 'Event rate ratio']])

# Clean: extract the PRAC-only slab

The source has three approval groups laid out as `(ratio, lo, hi)` triples:
columns 1–3 for *HCP and PRAC*, 4–6 for *HCP only*, 7–9 for *PRAC only*.
We take only the last triple plus the covariate column.


In [3]:
# PRAC only lives in columns 7, 8, 9. Column 0 is the covariate label.
PRAC_COLS = [0, 7, 8, 9]

# Data rows in the sheet (0-indexed). Row 0 is the title, rows 2-3 are
# headers, rows 4-51 are the body, rows 52+ are footnotes.
DATA_ROW_START = 4
DATA_ROW_END = 52  # exclusive

# Source-label fixes. The AIHW xlsx labels the age-group block as
# 'Mood disorders (all years)', but its values (50-69, 70-79, 80-89,
# 90+) are clearly age groups, and the report's Figure 8 labels this
# row as 'Age'. Treat it as a publishing error in the supplementary
# table and remap to 'Age' so the calculator surfaces the right label.
CATEGORY_RELABELS = {
    "Mood disorders (all years)": "Age",
}


def clean_table_s2_prac(input_path: Path):
    """Read Table S2 and reshape the PRAC-only slab into long format.

    Returns (DataFrame, summary_dict). The DataFrame has one row per
    (category, value) pair, with reference rows kept (event_rate_ratio = 1,
    NULL confidence intervals).
    """
    raw = pd.read_excel(input_path, sheet_name="Table S2", header=None)

    slab = raw.iloc[DATA_ROW_START:DATA_ROW_END, PRAC_COLS].copy()
    slab.columns = ["label", "event_rate_ratio", "lower_ci", "upper_ci"]
    slab = slab.reset_index(drop=True)

    # Coerce numerics. NaN in event_rate_ratio = category header row.
    for c in ["event_rate_ratio", "lower_ci", "upper_ci"]:
        slab[c] = pd.to_numeric(slab[c], errors="coerce")

    # Normalize whitespace on the label column.
    slab["label"] = (
        slab["label"].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Forward-fill the category onto each value row.
    is_category_header = slab["event_rate_ratio"].isna()
    slab["category"] = slab["label"].where(is_category_header).ffill()

    # Apply known source-label fixes (see CATEGORY_RELABELS above).
    slab["category"] = slab["category"].replace(CATEGORY_RELABELS)

    # Drop header rows; what remains are the value rows.
    values = slab.loc[~is_category_header].copy()
    values = values.rename(columns={"label": "value"})

    # Reference rows have event_rate_ratio == 1 and no CIs in the source.
    # Compare with a tolerance because the column is float-typed.
    values["is_reference"] = (values["event_rate_ratio"] - 1.0).abs() < 1e-9

    values = values[[
        "category", "value", "event_rate_ratio", "lower_ci", "upper_ci",
        "is_reference",
    ]].reset_index(drop=True)

    summary = {
        "input_rows_scanned": int(DATA_ROW_END - DATA_ROW_START),
        "category_headers": int(is_category_header.sum()),
        "value_rows": len(values),
        "categories": int(values["category"].nunique()),
        "reference_rows": int(values["is_reference"].sum()),
    }
    return values, summary


In [4]:
prac_df, summary = clean_table_s2_prac(INPUT_XLSX)
summary

{'input_rows_scanned': 48,
 'category_headers': 14,
 'value_rows': 34,
 'categories': 14,
 'reference_rows': 14}

In [8]:
prac_df

,category,value,event_rate_ratio,lower_ci,upper_ci,is_reference
0,Sex,Men,1.000,NaN,NaN,True
1,Sex,Women,1.030,0.998,1.063,False
2,Age,90+,1.000,NaN,NaN,True
3,Age,50–69,0.750,0.699,0.805,False
4,Age,70–79,0.818,0.781,0.856,False
5,Age,80–89,0.929,0.893,0.966,False
6,First Nations status,Non-Indigenous,1.000,NaN,NaN,True
7,First Nations status,First Nations,0.847,0.754,0.952,False
8,Culturally and linguistically diverse (CALD),Non-CALD,1.000,NaN,NaN,True
9,Culturally and linguistically diverse (CALD),CALD,0.924,0.875,0.976,False


In [6]:
# Sanity checks
assert len(prac_df) == 34, f"Expected 34 value rows, got {len(prac_df)}"
assert prac_df["is_reference"].sum() == prac_df["category"].nunique(), \
    "Each category must have exactly one reference row"
assert prac_df.loc[prac_df["is_reference"], ["lower_ci", "upper_ci"]].isna().all().all(), \
    "Reference rows must have NULL CIs"
assert prac_df.loc[~prac_df["is_reference"], ["lower_ci", "upper_ci"]].notna().all().all(), \
    "Non-reference rows must have both CIs"
assert prac_df.duplicated(["category", "value"]).sum() == 0, \
    "(category, value) must be unique"
print("All checks passed.")

All checks passed.


# Insert into DB

Connect, create the table from its schema file, then insert.


In [9]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv(".env.db", override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
with open("schemas/aged_care_timeliness_prac.sql", "r", encoding="utf-8") as f:
    schema_sql = f.read()
with engine.begin() as conn:
    conn.execute(text(schema_sql))
print("Applied schemas/aged_care_timeliness_prac.sql")
print(DB_NAME)

Applied schemas/aged_care_timeliness_prac.sql
carelink


In [10]:
# -----------------------------
# 3. INSERT INTO aged_care_timeliness_prac
# -----------------------------
# Idempotent re-runs: clear the table first. There are only 34 rows and the
# source data is fully replaced each time the AIHW publishes a new edition.
target_cols = [
    "category", "value", "event_rate_ratio",
    "lower_ci", "upper_ci", "is_reference",
]
to_insert = prac_df[target_cols].copy()

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE aged_care_timeliness_prac"))

to_insert.to_sql(
    "aged_care_timeliness_prac",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
)

print(f"Inserted {len(to_insert)} rows into aged_care_timeliness_prac")

Inserted 34 rows into aged_care_timeliness_prac


In [11]:
# -----------------------------
# 4. VALIDATE
# -----------------------------
with engine.begin() as conn:
    row_count = conn.execute(
        text("SELECT COUNT(*) FROM aged_care_timeliness_prac")
    ).scalar()
    ref_count = conn.execute(
        text("SELECT COUNT(*) FROM aged_care_timeliness_prac WHERE is_reference")
    ).scalar()
    cat_count = conn.execute(
        text("SELECT COUNT(DISTINCT category) FROM aged_care_timeliness_prac")
    ).scalar()

print(f"aged_care_timeliness_prac: {row_count} rows, "
      f"{cat_count} categories, {ref_count} reference rows")

# Spot-check a few known values from the source.
sample = pd.read_sql(
    """
    SELECT category, value, event_rate_ratio, lower_ci, upper_ci
    FROM aged_care_timeliness_prac
    WHERE (category = 'Dementia status' AND value = 'Dementia')
       OR (category = 'Living arrangement' AND value = 'Lives alone')
       OR (category = 'Priority level' AND value = 'High')
    ORDER BY category, value
    """,
    engine,
)
sample

aged_care_timeliness_prac: 34 rows, 14 categories, 14 reference rows


,category,value,event_rate_ratio,lower_ci,upper_ci
0,Dementia status,Dementia,1.553,1.500,1.609
1,Living arrangement,Lives alone,1.226,1.186,1.267
2,Priority level,High,1.498,1.417,1.584
